# Mikroskobik hücre sayımı — Ara sınav çalışması

Öğrenci no: **264309500** | Seed: **11526** | Veri: **Fluo-N2DH-GOWT1 (training)**

Önce geliştirme deneylerini, ardından doğrulamayı çalıştırıyorum. Test tahminleri birinci aşamada kaydediliyor; gerçek test sayıları yalnızca kilit commit'i doğrulandıktan sonra okunuyor.


## 0. Projenin hazırlanması

Google Colab yeterlidir, bilgisayara Jupyter kurmaya gerek yoktur. Bu notebook'u açın ve proje ZIP'ini istendiğinde yükleyin. ZIP içinde daha önce hesaplanmış sonuçlar bulunduğundan yeniden üretim için ayrı, temiz bir klasör oluşturulur.


In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess
PROJECT = Path.cwd()
if not (PROJECT / 'analysis_core.py').exists():
    matches = list(PROJECT.glob('*/analysis_core.py'))
    if matches:
        PROJECT = matches[0].parent
    else:
        archive = PROJECT / 'Goruntu_Isleme_Projesi_264309500.zip'
        if not archive.exists():
            try:
                from google.colab import files
                print('Proje zip dosyasını seçin:')
                uploaded = files.upload()
                archive = PROJECT / next(x for x in uploaded if x.endswith('.zip'))
            except Exception as exc:
                raise RuntimeError('Proje dosyalarını notebook ile aynı dizine koyun veya zip dosyasını yükleyin.') from exc
        with zipfile.ZipFile(archive) as z:
            z.extractall(PROJECT)
        matches = list(PROJECT.glob('*/analysis_core.py'))
        if not matches:
            raise RuntimeError('Zip içinde analysis_core.py bulunamadı.')
        PROJECT = matches[0].parent
# Sonuç dosyalarıyla gelen paketi yeniden çalıştırırken kayıtları koru.
if (PROJECT / 'test_predictions_locked.csv').is_file():
    import shutil
    fresh = PROJECT.parent / (PROJECT.name + '_yeniden_calisma')
    if fresh.exists():
        raise RuntimeError('Yeniden çalışma klasörü zaten var. Karışıklığı önlemek için yeni bir Colab oturumu açın.')
    fresh.mkdir()
    for name in ['analysis_core.py','make_report.py','requirements.txt','selected_frames.csv',
                 'main.ipynb','.gitignore','README.md']:
        shutil.copy2(PROJECT / name, fresh / name)
    PROJECT = fresh
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('Çalışma klasörü:',PROJECT)
subprocess.check_call([sys.executable,'-m','pip','-q','install','-r','requirements.txt'])
import analysis_core as ac
print('Seed:',ac.SEED)

## 1. Eğitim verisinin hazırlanması

Cell Tracking Challenge'ın yalnızca eğitim arşivi kullanılıyor. `01` ve `02` görüntüleri ile bu sekansların GT klasörleri bu arşivde bulunuyor. İnternet bağlantısı yoksa arşivi `data/` klasörüne yüklemek yeterlidir.


In [ ]:
import urllib.request, shutil, zipfile
DATA_ROOT=PROJECT/'data'/'Fluo-N2DH-GOWT1'
DATA_DIR=PROJECT/'data'; DATA_DIR.mkdir(exist_ok=True)
archive=DATA_DIR/'Fluo-N2DH-GOWT1.zip'
if not (DATA_ROOT/'01').is_dir():
    if not archive.is_file():
        print('Yalnızca training zip indiriliyor:',ac.DATASET_URL)
        with urllib.request.urlopen(ac.DATASET_URL, timeout=120) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
    with zipfile.ZipFile(archive,'r') as z:
        z.extractall(DATA_DIR)
ds=ac.discover_dataset(DATA_ROOT)
print('Kare sayıları (01/02):',len(ds.frames01),len(ds.frames02))
print('Şekil / veri tipi:',ds.image('01',min(ds.frames01)).shape, ds.image('01',min(ds.frames01)).dtype)

## 2. Görüntülerin seçimi

Seed 11526 ile seçilmiş dosya adları `selected_frames.csv` içinde kayıtlı. Kod, bu seçimin diskteki görüntülerle tutarlı olduğunu kontrol eder.


In [ ]:
import pandas as pd
selected=ac.frame_selection(ds,PROJECT)
print(selected.to_string(index=False))
print('Geliştirme / doğrulama / test:',selected['group'].value_counts().to_dict())

## 3. İlk aşama: geliştirme ve doğrulama

On geliştirme karesinde 24 ayar denenir. En iyi üçü beş doğrulama karesinde ölçülür. Seçilen ayar ve 15 test tahmini kaydedilir; bu aşamada test etiketleri okunmaz.


In [ ]:
from pprint import pprint
stage1=ac.run_stage1(DATA_ROOT,PROJECT)
pprint(stage1)
print('Üç aday / doğrulama:')
display(pd.read_csv('validation_results.csv'))
print('Deney tablosu (ilk 24):')
display(pd.read_csv('experiments.csv'))
print('Test tahminleri kaydedildi; referans sayılar ikinci aşamada okunacak.')

## 4. Test tahminlerinin Git'e kaydedilmesi

`final_pipeline.json` ve `test_predictions_locked.csv` dosyalarını test etiketlerini incelemeden ayrı bir commit'te saklamak gerekiyor. Aşağıdaki hücre yerel commit oluşturur. Ödevin GitHub koşulu için bu commit ayrıca kendi deponuza gönderilmelidir.


In [ ]:
import subprocess, os
if not (PROJECT/'.git').is_dir():
    subprocess.check_call(['git','init'],cwd=PROJECT)
# Git kimliğiniz Colab içinde yoksa geçici ve açıkça teknik bir yerel kimlik kullanılır.
# GitHub hesabınıza ait bilgileri girmek isterseniz önceden git config ile değiştirin.
for field,value in [('user.name','Student 264309500'),('user.email','student264309500@local.invalid')]:
    existing=subprocess.run(['git','config','--get',field],cwd=PROJECT,capture_output=True,text=True)
    if existing.returncode!=0:
        subprocess.check_call(['git','config',field,value],cwd=PROJECT)
required=['README.md','requirements.txt','main.ipynb','analysis_core.py','make_report.py',
          '.gitignore','selected_frames.csv','dataset_diagnostics.csv','analysis_environment.json',
          'development_validation_reference.csv','development_per_frame.csv','experiments.csv',
          'validation_results.csv','validation_per_frame.csv','final_pipeline.json',
          'test_predictions_locked.csv','lock_manifest.json']
subprocess.check_call(['git','add',*required],cwd=PROJECT)
subprocess.check_call(['git','commit','-m','stage1: lock test predictions before labels'],cwd=PROJECT)
print('Aşama 1 kilit commit oluşturuldu. GitHub remote ekleme/push işlemi ayrıca yapılmalı.')
print('git remote add origin https://github.com/KULLANICI/REPO.git')
print('git branch -M main && git push -u origin main')

### 4.1. Kilit commit'inin kontrolü

Birinci commit'in kimliğini alıp dosyaların o kayıttaki sürümleriyle aynı olup olmadığını kontrol ediyorum.


In [ ]:
LOCK_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip()
ac.verify_lock_commit(PROJECT,LOCK_COMMIT)
print('Kilit commit hash:',LOCK_COMMIT)
print('Yerel içerik commit ile eşleşti. GitHub push işlemini ayrıca doğrulayın.')

## 5. İkinci aşama: bağımsız test

Bu aşamada Sequence 02 referanslarına bakılarak sayım hataları hesaplanır. İlk aşamadaki seçim ve parametreler değiştirilmez.


In [ ]:
stage2=ac.run_stage2(DATA_ROOT,PROJECT,LOCK_COMMIT)
pprint(stage2)
display(pd.read_csv('test_results.csv'))
display(pd.read_csv('test_metrics.csv'))

## 6. Teknik rapor

CSV sonuçları, işlem örnekleri ve zamansal grafikler kullanılarak sekiz sayfalık ana rapor ve ayrı kaynakça hazırlanır. İşlem sürelerinin kullanılan bilgisayara göre değişebileceği unutulmamalıdır.


In [ ]:
from make_report import build
report_path=build(PROJECT)
print('Nihai rapor:',report_path)
print('Özel teslim ortamı için bilgisayarınıza indirin. TIFF görselleri GitHub üzerinden paylaşılmamalı.')
try:
    from google.colab import files
    files.download(str(report_path))
except ImportError:
    pass

## 7. Sonuçların kaydedilmesi

Test değerlendirmesini ikinci bir commit'e ekleyin; daha sonra GitHub'a gönderin. Özel teslim için PDF raporu ve görselleri kullanın. Orijinal mikroskop görüntüleri, anotasyonlar ve görsel çıktı gömülü notebook dosyaları GitHub'a yüklenmemelidir.


In [ ]:
# Test sonuçlarını ayrı bir commit olarak kaydet.
subprocess.check_call(['git','add','test_results.csv','test_metrics.csv'],cwd=PROJECT)
subprocess.check_call(['git','commit','-m','stage2: independent test evaluation'],cwd=PROJECT)
print("Aşama 2 commit oluşturuldu. GitHub'a git push gönderin ve repository bağlantısını teslim ekranına ekleyin.")